# TicketForge sales and operations report

Read-only SQL against the two service databases: when people buy, how fast events sell, how many holds turn into orders, refunds, no-shows and price against sell-through.

Seed the stack first (`make seed`), then open it from the repository root:

```bash
uv run --with jupyter --with pandas --with matplotlib jupyter lab notebooks/sales_report.ipynb
```

Nothing here writes to the database.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

MYSQL_HOST = os.getenv("MYSQL_HOST", "127.0.0.1")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", "ticketforge")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "ticketforge")
CORE = os.getenv("MYSQL_DATABASE", "core")
CHECKOUT = os.getenv("CHECKOUT_DATABASE", "checkout")

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/?charset=utf8mb4",
    pool_pre_ping=True,
)


def query(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


pd.set_option("display.max_columns", 50)
print("connected to", MYSQL_HOST, CORE, "+", CHECKOUT)

In [ ]:
events = query(f"""
    SELECT e.id AS event_id, e.title, e.category, e.status, e.starts_at, e.ends_at,
           v.name AS venue, v.city
    FROM {CORE}.core_event e
    JOIN {CORE}.core_venue v ON v.id = e.venue_id
""")

ticket_types = query(f"""
    SELECT t.id AS ticket_type_id, t.event_id, t.name, t.price_cents, t.quantity_total,
           t.inventory_mode
    FROM {CORE}.core_ticket_type t
""")

orders = query(f"""
    SELECT o.id, o.user_id, o.event_id, o.total_cents, o.currency, o.status,
           o.created_at, o.hold_id
    FROM {CHECKOUT}.orders o
""")

tickets = query(f"""
    SELECT t.id, t.order_id, t.ticket_type_id, t.event_seat_id, t.status, t.checked_in_at
    FROM {CHECKOUT}.ticket t
""")

refunds = query(f"""
    SELECT r.id, r.order_id, r.amount_cents, r.status, r.reason, r.created_at
    FROM {CHECKOUT}.refund r
""")

holds = query(f"""
    SELECT h.id, h.user_id, h.event_id, h.status, h.created_at, h.expires_at
    FROM {CHECKOUT}.hold h
""")

orders["created_at"] = pd.to_datetime(orders["created_at"])
events["starts_at"] = pd.to_datetime(events["starts_at"])
print(f"{len(events)} events · {len(orders)} orders · {len(tickets)} tickets · {len(holds)} holds")

sales = orders[orders.status.isin(["paid", "partially_refunded"])].merge(
    events[["event_id", "title", "category", "city"]], on="event_id", how="left"
)
sales["revenue"] = sales["total_cents"] / 100

## 1. When do people buy?

In [ ]:
by_hour = (
    sales.assign(hour=sales.created_at.dt.hour)
    .groupby("hour")
    .agg(orders=("id", "count"), revenue=("revenue", "sum"))
)
by_weekday = (
    sales.assign(day=sales.created_at.dt.day_name())
    .groupby("day")
    .agg(orders=("id", "count"), revenue=("revenue", "sum"))
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
by_hour["orders"].plot(kind="bar", ax=axes[0], color="#f6b73c", title="Orders by hour (UTC)")
by_weekday.reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])[
    "revenue"
].plot(kind="bar", ax=axes[1], color="#78a9ff", title="Revenue by weekday (USD)")
plt.tight_layout()
plt.show()

## 2. How fast do events sell out?

In [ ]:
sold = tickets.merge(
    ticket_types[["ticket_type_id", "event_id", "name", "price_cents", "quantity_total"]],
    on="ticket_type_id",
    how="left",
)
sell_through = (
    sold.groupby(["event_id", "ticket_type_id"])
    .size()
    .rename("sold")
    .reset_index()
    .merge(ticket_types, on=["event_id", "ticket_type_id"], how="right")
    .fillna({"sold": 0})
)
sell_through["sell_through_pct"] = (sell_through.sold / sell_through.quantity_total * 100).round(1)
sell_through = sell_through.merge(events[["event_id", "title"]], on="event_id", how="left")
sell_through.sort_values("sell_through_pct", ascending=False).head(15)[
    ["title", "name", "sold", "quantity_total", "sell_through_pct"]
]

## 3. Checkout drop-off (holds that expire without payment)

In [ ]:
hold_stats = holds.groupby("status").size().rename("holds").reset_index()
hold_stats["share"] = (hold_stats.holds / hold_stats.holds.sum() * 100).round(1)

paid_holds = set(sales.hold_id.dropna())
holds_with_order = holds[holds.id.isin(paid_holds)]
conversion = len(holds_with_order) / len(holds) * 100 if len(holds) else 0

fig, ax = plt.subplots(figsize=(6, 4))
hold_stats.set_index("status")["holds"].plot(
    kind="bar", ax=ax, color="#f6b73c", title="Hold outcomes"
)
ax.set_ylabel("holds")
plt.tight_layout()
plt.show()
print(f"hold → paid conversion: {conversion:.1f}%")
hold_stats

## 4. Refund rate by category

In [ ]:
refunded = refunds.merge(
    sales[["id", "event_id", "category", "revenue"]],
    left_on="order_id",
    right_on="id",
    how="left",
    suffixes=("", "_order"),
)
refund_by_category = (
    refunded.groupby("category")
    .agg(refunds=("id", "count"), refunded_amount=("amount_cents", "sum"))
    .reset_index()
)
refund_by_category["refunded_amount"] = refund_by_category.refunded_amount / 100

orders_by_category = sales.groupby("category").size().rename("orders").reset_index()
refund_by_category = orders_by_category.merge(refund_by_category, on="category", how="left").fillna(
    0
)
refund_by_category["refund_rate_pct"] = (
    refund_by_category.refunds / refund_by_category.orders * 100
).round(2)
refund_by_category.sort_values("refund_rate_pct", ascending=False)

## 5. No-show rate

In [ ]:
ticket_counts = tickets.groupby("status").size().rename("tickets").reset_index()
issued = tickets[tickets.status.isin(["valid", "checked_in"])]
no_show_rate = (issued.status.eq("valid").sum() / len(issued) * 100) if len(issued) else 0

fig, ax = plt.subplots(figsize=(6, 4))
ticket_counts.set_index("status")["tickets"].plot(
    kind="bar", ax=ax, color="#3ddc97", title="Ticket statuses"
)
ax.set_ylabel("tickets")
plt.tight_layout()
plt.show()
print(f"no-show rate (unscanned valid tickets): {no_show_rate:.1f}%")
ticket_counts

## 6. Price vs sell-through

In [ ]:
scatter = sell_through.merge(events[["event_id", "title"]], on="event_id", how="left")
scatter["price"] = scatter.price_cents / 100

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(scatter.price, scatter.sell_through_pct, c="#f6b73c", alpha=0.75)
ax.set_xlabel("price (USD)")
ax.set_ylabel("sell-through (%)")
ax.set_title("Price vs sell-through")
plt.tight_layout()
plt.show()
scatter[["price", "sell_through_pct"]].corr(numeric_only=True)

## Summary

Fill this in after running the notebook against your own data:

- Peak purchase window:
- Average hold → paid conversion:
- Best-selling category:
- Refund hot-spots:
- No-show rate:
- Price sensitivity:

Export this notebook with **File → Export → HTML** (or `jupyter nbconvert --to pdf`) to attach the written summary required by the spec.
